# 01 Load Silver

Loads the 23 AdventureWorks files (the **bronze** layer, in `Files/AdventureWorks`) into typed Delta tables in the `silver` schema, then checks the result.

The `00_Setup` notebook runs this for you. To run it on its own:

1. Make sure the files are in `Files/AdventureWorks` of a schema-enabled Lakehouse.
2. Set `LAKEHOUSE_NAME` below to that Lakehouse (it must be in this workspace).
3. Run all cells. You do not need to attach a Lakehouse to the notebook.

**File format** (the same for every file): pipe `|` delimited, UTF-8, header row, no quoting, empty field = null, dates as `yyyy-MM-dd`, booleans as `true` / `false`.

In [ ]:
# Parameters. A pipeline or the setup notebook can override these.
LAKEHOUSE_NAME = "AdventureWorksLH"

In [ ]:
import notebookutils

lakehouse = notebookutils.lakehouse.get(LAKEHOUSE_NAME)
ROOT = f"abfss://{lakehouse.workspaceId}@onelake.dfs.fabric.microsoft.com/{lakehouse.id}"
SOURCE = f"{ROOT}/Files/AdventureWorks"
SILVER = f"{ROOT}/Tables/silver"

# V-Order and optimized write make the tables faster for Direct Lake and the SQL analytics endpoint.
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")


def ensure_schema(schema):
    """Create the schema in the Lakehouse if it is not there yet."""
    workspace = notebookutils.runtime.context["currentWorkspaceName"]
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{workspace}`.`{LAKEHOUSE_NAME}`.`{schema}`")
    except Exception as error:
        # Writing a table under Tables/<schema> also creates the schema, so carry on.
        print(f"Could not create schema {schema} with Spark SQL ({type(error).__name__}); it will be created by the first write.")


ensure_schema("silver")

## Table definitions

In [ ]:
# One entry per table: explicit schema, primary key and expected row count.
TABLES = {
    "DimAccount": {
        "key": ["AccountKey"],
        "rows": 99,
        "schema": """
            AccountKey INT,
            ParentAccountKey INT,
            AccountCodeAlternateKey INT,
            ParentAccountCodeAlternateKey INT,
            AccountDescription STRING,
            AccountType STRING,
            Operator STRING,
            CustomMembers STRING,
            ValueType STRING,
            CustomMemberOptions STRING
        """,
    },
    "DimCurrency": {
        "key": ["CurrencyKey"],
        "rows": 105,
        "schema": """
            CurrencyKey INT,
            CurrencyAlternateKey STRING,
            CurrencyName STRING
        """,
    },
    "DimCustomer": {
        "key": ["CustomerKey"],
        "rows": 18484,
        "schema": """
            CustomerKey INT,
            GeographyKey INT,
            CustomerAlternateKey STRING,
            Title STRING,
            FirstName STRING,
            MiddleName STRING,
            LastName STRING,
            NameStyle BOOLEAN,
            BirthDate DATE,
            MaritalStatus STRING,
            Suffix STRING,
            Gender STRING,
            EmailAddress STRING,
            YearlyIncome DECIMAL(19,4),
            TotalChildren INT,
            NumberChildrenAtHome INT,
            EnglishEducation STRING,
            EnglishOccupation STRING,
            HouseOwnerFlag STRING,
            NumberCarsOwned INT,
            AddressLine1 STRING,
            AddressLine2 STRING,
            Phone STRING,
            DateFirstPurchase DATE,
            CommuteDistance STRING
        """,
    },
    "DimDate": {
        "key": ["DateKey"],
        "rows": 3652,
        "schema": """
            DateKey INT,
            FullDateAlternateKey DATE,
            DayNumberOfWeek INT,
            EnglishDayNameOfWeek STRING,
            DayNumberOfMonth INT,
            DayNumberOfYear INT,
            WeekNumberOfYear INT,
            EnglishMonthName STRING,
            MonthNumberOfYear INT,
            CalendarQuarter INT,
            CalendarYear INT,
            CalendarSemester INT,
            FiscalQuarter INT,
            FiscalYear INT,
            FiscalSemester INT
        """,
    },
    "DimDepartmentGroup": {
        "key": ["DepartmentGroupKey"],
        "rows": 7,
        "schema": """
            DepartmentGroupKey INT,
            ParentDepartmentGroupKey INT,
            DepartmentGroupName STRING
        """,
    },
    "DimEmployee": {
        "key": ["EmployeeKey"],
        "rows": 296,
        "schema": """
            EmployeeKey INT,
            ParentEmployeeKey INT,
            EmployeeNationalIDAlternateKey STRING,
            ParentEmployeeNationalIDAlternateKey STRING,
            SalesTerritoryKey INT,
            FirstName STRING,
            LastName STRING,
            MiddleName STRING,
            NameStyle BOOLEAN,
            Title STRING,
            HireDate DATE,
            BirthDate DATE,
            LoginID STRING,
            EmailAddress STRING,
            Phone STRING,
            MaritalStatus STRING,
            EmergencyContactName STRING,
            EmergencyContactPhone STRING,
            SalariedFlag BOOLEAN,
            Gender STRING,
            PayFrequency INT,
            BaseRate DECIMAL(19,4),
            VacationHours INT,
            SickLeaveHours INT,
            CurrentFlag BOOLEAN,
            SalesPersonFlag BOOLEAN,
            DepartmentName STRING,
            StartDate DATE,
            EndDate DATE,
            Status STRING
        """,
    },
    "DimGeography": {
        "key": ["GeographyKey"],
        "rows": 655,
        "schema": """
            GeographyKey INT,
            City STRING,
            StateProvinceCode STRING,
            StateProvinceName STRING,
            CountryRegionCode STRING,
            EnglishCountryRegionName STRING,
            PostalCode STRING,
            SalesTerritoryKey INT,
            IpAddressLocator STRING
        """,
    },
    "DimOrganization": {
        "key": ["OrganizationKey"],
        "rows": 14,
        "schema": """
            OrganizationKey INT,
            ParentOrganizationKey INT,
            PercentageOfOwnership STRING,
            OrganizationName STRING,
            CurrencyKey INT
        """,
    },
    "DimProduct": {
        "key": ["ProductKey"],
        "rows": 606,
        "schema": """
            ProductKey INT,
            ProductAlternateKey STRING,
            ProductSubcategoryKey INT,
            WeightUnitMeasureCode STRING,
            SizeUnitMeasureCode STRING,
            EnglishProductName STRING,
            StandardCost DECIMAL(19,4),
            FinishedGoodsFlag BOOLEAN,
            Color STRING,
            SafetyStockLevel INT,
            ReorderPoint INT,
            ListPrice DECIMAL(19,4),
            Size STRING,
            SizeRange STRING,
            Weight DOUBLE,
            DaysToManufacture INT,
            ProductLine STRING,
            DealerPrice DECIMAL(19,4),
            Class STRING,
            Style STRING,
            ModelName STRING,
            EnglishDescription STRING,
            StartDate DATE,
            EndDate DATE,
            Status STRING
        """,
    },
    "DimProductCategory": {
        "key": ["ProductCategoryKey"],
        "rows": 4,
        "schema": """
            ProductCategoryKey INT,
            ProductCategoryAlternateKey INT,
            EnglishProductCategoryName STRING
        """,
    },
    "DimProductSubcategory": {
        "key": ["ProductSubcategoryKey"],
        "rows": 37,
        "schema": """
            ProductSubcategoryKey INT,
            ProductSubcategoryAlternateKey INT,
            EnglishProductSubcategoryName STRING,
            ProductCategoryKey INT
        """,
    },
    "DimPromotion": {
        "key": ["PromotionKey"],
        "rows": 16,
        "schema": """
            PromotionKey INT,
            PromotionAlternateKey INT,
            EnglishPromotionName STRING,
            DiscountPct DOUBLE,
            EnglishPromotionType STRING,
            EnglishPromotionCategory STRING,
            StartDate DATE,
            EndDate DATE,
            MinQty INT,
            MaxQty INT
        """,
    },
    "DimReseller": {
        "key": ["ResellerKey"],
        "rows": 701,
        "schema": """
            ResellerKey INT,
            GeographyKey INT,
            ResellerAlternateKey STRING,
            Phone STRING,
            BusinessType STRING,
            ResellerName STRING,
            NumberEmployees INT,
            OrderFrequency STRING,
            OrderMonth INT,
            FirstOrderYear INT,
            LastOrderYear INT,
            ProductLine STRING,
            AddressLine1 STRING,
            AddressLine2 STRING,
            AnnualSales DECIMAL(19,4),
            BankName STRING,
            MinPaymentType INT,
            MinPaymentAmount DECIMAL(19,4),
            AnnualRevenue DECIMAL(19,4),
            YearOpened INT
        """,
    },
    "DimSalesTerritory": {
        "key": ["SalesTerritoryKey"],
        "rows": 11,
        "schema": """
            SalesTerritoryKey INT,
            SalesTerritoryAlternateKey INT,
            SalesTerritoryRegion STRING,
            SalesTerritoryCountry STRING,
            SalesTerritoryGroup STRING
        """,
    },
    "DimScenario": {
        "key": ["ScenarioKey"],
        "rows": 3,
        "schema": """
            ScenarioKey INT,
            ScenarioName STRING
        """,
    },
    "FactCallCenter": {
        "key": ["FactCallCenterID"],
        "rows": 120,
        "schema": """
            FactCallCenterID INT,
            DateKey INT,
            WageType STRING,
            Shift STRING,
            LevelOneOperators INT,
            LevelTwoOperators INT,
            TotalOperators INT,
            Calls INT,
            AutomaticResponses INT,
            Orders INT,
            IssuesRaised INT,
            AverageTimePerIssue INT,
            ServiceGrade DOUBLE,
            Date DATE
        """,
    },
    "FactCurrencyRate": {
        "key": ["CurrencyKey", "DateKey"],
        "rows": 14264,
        "schema": """
            CurrencyKey INT,
            DateKey INT,
            AverageRate DOUBLE,
            EndOfDayRate DOUBLE,
            Date DATE
        """,
    },
    "FactFinance": {
        "key": ["FinanceKey"],
        "rows": 39409,
        "schema": """
            FinanceKey INT,
            DateKey INT,
            OrganizationKey INT,
            DepartmentGroupKey INT,
            ScenarioKey INT,
            AccountKey INT,
            Amount DOUBLE,
            Date DATE
        """,
    },
    "FactInternetSales": {
        "key": ["SalesOrderNumber", "SalesOrderLineNumber"],
        "rows": 60398,
        "schema": """
            ProductKey INT,
            OrderDateKey INT,
            DueDateKey INT,
            ShipDateKey INT,
            CustomerKey INT,
            PromotionKey INT,
            CurrencyKey INT,
            SalesTerritoryKey INT,
            SalesOrderNumber STRING,
            SalesOrderLineNumber INT,
            RevisionNumber INT,
            OrderQuantity INT,
            UnitPrice DECIMAL(19,4),
            ExtendedAmount DECIMAL(19,4),
            UnitPriceDiscountPct DOUBLE,
            DiscountAmount DOUBLE,
            ProductStandardCost DECIMAL(19,4),
            TotalProductCost DECIMAL(19,4),
            SalesAmount DECIMAL(19,4),
            TaxAmt DECIMAL(19,4),
            Freight DECIMAL(19,4),
            CarrierTrackingNumber STRING,
            CustomerPONumber STRING,
            OrderDate DATE,
            DueDate DATE,
            ShipDate DATE
        """,
    },
    "FactProductInventory": {
        "key": ["ProductKey", "DateKey"],
        "rows": 776286,
        "schema": """
            ProductKey INT,
            DateKey INT,
            MovementDate DATE,
            UnitCost DECIMAL(19,4),
            UnitsIn INT,
            UnitsOut INT,
            UnitsBalance INT
        """,
    },
    "FactResellerSales": {
        "key": ["SalesOrderNumber", "SalesOrderLineNumber"],
        "rows": 60855,
        "schema": """
            ProductKey INT,
            OrderDateKey INT,
            DueDateKey INT,
            ShipDateKey INT,
            ResellerKey INT,
            EmployeeKey INT,
            PromotionKey INT,
            CurrencyKey INT,
            SalesTerritoryKey INT,
            SalesOrderNumber STRING,
            SalesOrderLineNumber INT,
            RevisionNumber INT,
            OrderQuantity INT,
            UnitPrice DECIMAL(19,4),
            ExtendedAmount DECIMAL(19,4),
            UnitPriceDiscountPct DOUBLE,
            DiscountAmount DOUBLE,
            ProductStandardCost DECIMAL(19,4),
            TotalProductCost DECIMAL(19,4),
            SalesAmount DECIMAL(19,4),
            TaxAmt DECIMAL(19,4),
            Freight DECIMAL(19,4),
            CarrierTrackingNumber STRING,
            CustomerPONumber STRING,
            OrderDate DATE,
            DueDate DATE,
            ShipDate DATE
        """,
    },
    "FactSalesQuota": {
        "key": ["SalesQuotaKey"],
        "rows": 163,
        "schema": """
            SalesQuotaKey INT,
            EmployeeKey INT,
            DateKey INT,
            CalendarYear INT,
            CalendarQuarter INT,
            SalesAmountQuota DECIMAL(19,4),
            Date DATE
        """,
    },
    "FactSurveyResponse": {
        "key": ["SurveyResponseKey"],
        "rows": 2727,
        "schema": """
            SurveyResponseKey INT,
            DateKey INT,
            CustomerKey INT,
            ProductCategoryKey INT,
            EnglishProductCategoryName STRING,
            ProductSubcategoryKey INT,
            EnglishProductSubcategoryName STRING,
            Date DATE
        """,
    },
}

In [ ]:
# (child table, child column, parent table, parent column)
RELATIONSHIPS = [
    ("DimAccount", "ParentAccountKey", "DimAccount", "AccountKey"),
    ("DimCustomer", "GeographyKey", "DimGeography", "GeographyKey"),
    ("DimDepartmentGroup", "ParentDepartmentGroupKey", "DimDepartmentGroup", "DepartmentGroupKey"),
    ("DimEmployee", "ParentEmployeeKey", "DimEmployee", "EmployeeKey"),
    ("DimEmployee", "SalesTerritoryKey", "DimSalesTerritory", "SalesTerritoryKey"),
    ("DimGeography", "SalesTerritoryKey", "DimSalesTerritory", "SalesTerritoryKey"),
    ("DimOrganization", "ParentOrganizationKey", "DimOrganization", "OrganizationKey"),
    ("DimOrganization", "CurrencyKey", "DimCurrency", "CurrencyKey"),
    ("DimProduct", "ProductSubcategoryKey", "DimProductSubcategory", "ProductSubcategoryKey"),
    ("DimProductSubcategory", "ProductCategoryKey", "DimProductCategory", "ProductCategoryKey"),
    ("DimReseller", "GeographyKey", "DimGeography", "GeographyKey"),
    ("FactCallCenter", "DateKey", "DimDate", "DateKey"),
    ("FactCurrencyRate", "CurrencyKey", "DimCurrency", "CurrencyKey"),
    ("FactCurrencyRate", "DateKey", "DimDate", "DateKey"),
    ("FactFinance", "DateKey", "DimDate", "DateKey"),
    ("FactFinance", "OrganizationKey", "DimOrganization", "OrganizationKey"),
    ("FactFinance", "DepartmentGroupKey", "DimDepartmentGroup", "DepartmentGroupKey"),
    ("FactFinance", "ScenarioKey", "DimScenario", "ScenarioKey"),
    ("FactFinance", "AccountKey", "DimAccount", "AccountKey"),
    ("FactInternetSales", "ProductKey", "DimProduct", "ProductKey"),
    ("FactInternetSales", "OrderDateKey", "DimDate", "DateKey"),
    ("FactInternetSales", "DueDateKey", "DimDate", "DateKey"),
    ("FactInternetSales", "ShipDateKey", "DimDate", "DateKey"),
    ("FactInternetSales", "CustomerKey", "DimCustomer", "CustomerKey"),
    ("FactInternetSales", "PromotionKey", "DimPromotion", "PromotionKey"),
    ("FactInternetSales", "CurrencyKey", "DimCurrency", "CurrencyKey"),
    ("FactInternetSales", "SalesTerritoryKey", "DimSalesTerritory", "SalesTerritoryKey"),
    ("FactProductInventory", "ProductKey", "DimProduct", "ProductKey"),
    ("FactProductInventory", "DateKey", "DimDate", "DateKey"),
    ("FactResellerSales", "ProductKey", "DimProduct", "ProductKey"),
    ("FactResellerSales", "OrderDateKey", "DimDate", "DateKey"),
    ("FactResellerSales", "DueDateKey", "DimDate", "DateKey"),
    ("FactResellerSales", "ShipDateKey", "DimDate", "DateKey"),
    ("FactResellerSales", "ResellerKey", "DimReseller", "ResellerKey"),
    ("FactResellerSales", "EmployeeKey", "DimEmployee", "EmployeeKey"),
    ("FactResellerSales", "PromotionKey", "DimPromotion", "PromotionKey"),
    ("FactResellerSales", "CurrencyKey", "DimCurrency", "CurrencyKey"),
    ("FactResellerSales", "SalesTerritoryKey", "DimSalesTerritory", "SalesTerritoryKey"),
    ("FactSalesQuota", "EmployeeKey", "DimEmployee", "EmployeeKey"),
    ("FactSalesQuota", "DateKey", "DimDate", "DateKey"),
    ("FactSurveyResponse", "DateKey", "DimDate", "DateKey"),
    ("FactSurveyResponse", "CustomerKey", "DimCustomer", "CustomerKey"),
    ("FactSurveyResponse", "ProductCategoryKey", "DimProductCategory", "ProductCategoryKey"),
    ("FactSurveyResponse", "ProductSubcategoryKey", "DimProductSubcategory", "ProductSubcategoryKey"),
]

## Load

In [ ]:
def read_table(name):
    """Read one file with its explicit schema. Fails loudly on a bad row or a header that does not match."""
    return (
        spark.read.format("csv")
        .option("header", "true")
        .option("sep", "|")
        .option("quote", "")              # no quoting: values such as 1" headset are literal text
        .option("encoding", "UTF-8")
        .option("dateFormat", "yyyy-MM-dd")
        .option("mode", "FAILFAST")
        .option("enforceSchema", "false")  # check the header names against the schema
        .schema(TABLES[name]["schema"])
        .load(f"{SOURCE}/{name}.csv")
    )


for name in TABLES:
    (
        read_table(name).write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(f"{SILVER}/{name}")
    )
    print(f"Loaded silver.{name}")

## Validate

In [ ]:
# Validate what was written: row counts, unique keys, and every relationship.
problems = []
loaded = {name: spark.read.format("delta").load(f"{SILVER}/{name}") for name in TABLES}

for name, spec in TABLES.items():
    df = loaded[name]
    count = df.count()
    if count != spec["rows"]:
        problems.append(f"{name}: expected {spec['rows']} rows, found {count}")
    if df.select(*spec["key"]).distinct().count() != count:
        problems.append(f"{name}: key {spec['key']} is not unique")
    if df.dropna(subset=spec["key"]).count() != count:
        problems.append(f"{name}: key {spec['key']} contains nulls")

for child, child_col, parent, parent_col in RELATIONSHIPS:
    orphans = (
        loaded[child].select(child_col).where(f"{child_col} IS NOT NULL").distinct()
        .join(loaded[parent].select(parent_col).distinct(), loaded[child][child_col] == loaded[parent][parent_col], "left_anti")
        .count()
    )
    if orphans:
        problems.append(f"{child}.{child_col}: {orphans} values not found in {parent}.{parent_col}")

if problems:
    raise Exception("Validation failed:\n" + "\n".join(problems))
print(f"All {len(TABLES)} silver tables loaded and validated. BEST DAY EVER!!!")

Next: `02_Build_Gold` turns these tables into a star schema with current dates.

Thanks to MIM for help with the original load code.